# Extract "zero features"
Zero features are considered to be the operations whose average value across all time series, regardelss of the process, and over all the time series length is lower than a threshold compatible with numerical errors  

In [1]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import sys
import pandas as pd

In [2]:
plt.style.use("my_style.mplstyle")

In [ ]:
cwd = os.getcwd()

path = os.path.abspath(os.path.join(cwd, os.pardir))
print(path)
# Directories to save data 
dir_zero= path + '/data-zero/'

# Directory for supplementary materials
dir_supp= path + '/supplementary-material/'

# Directory to find hctsa pre-processed hctsa data
dir_hctsa= path + '/data-tr/main-analysis/data-analysis/'


/Users/tdal0054/Desktop/Trial3/Time-reversibility/src/main


In [11]:
# Load the data
df_hctsa=pd.read_csv(dir_hctsa+'df_TS_DataMat_diff.csv')
df_hctsa.set_index(['Model'], inplace=True)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/tdal0054/Desktop/Trial3/Time-reversibility/src/data-tr/data-tr/main-analysis/data-analysis/df_TS_DataMat_diff.csv'

In [ ]:
# Compute the mean of the features across all time series and models
ser_mean_hctsa=df_hctsa.mean()
ser_mean_sorted_hctsa=ser_mean_hctsa.sort_values(ascending=True)
df_mean_sorted_hctsa=pd.DataFrame(ser_mean_sorted_hctsa).reset_index()
df_mean_sorted_hctsa=df_mean_sorted_hctsa.rename(columns={'index':'Name',0:'Value'})

# Create the save directory if it doesn't exist
os.makedirs(dir_zero, exist_ok=True)

# Save data
df_mean_sorted_hctsa.to_csv(dir_zero+'df_ops_sorted.csv', index=False)

In [ ]:
# Distinguish between zero and non-zero values with threshold (consider magnitude of the values)
threshold=5.*sys.float_info.epsilon

df_ops_nonzero=df_mean_sorted_hctsa[abs(df_mean_sorted_hctsa['Value'])>threshold]
df_ops_zero= df_mean_sorted_hctsa[abs(df_mean_sorted_hctsa['Value'])<=threshold]

# Save data
df_ops_nonzero.to_csv(dir_zero+'df_ops_nonzero.csv', index=False)
df_ops_zero.to_csv(dir_zero+'df_ops_zero.csv', index=False)

In [ ]:
fig, ax = plt.subplots()

geometric_sequence = [pow(10.0, k) for k in np.arange(-20, 6, 1)]      # bins with logarithic scale -> to visualize data that span on different scales
#geometric_sequence = [-x for x in geometric_sequence] + geometric_sequence  # Add negative values
#geometric_sequence = sorted(geometric_sequence)
hist = ax.hist(abs(df_mean_sorted_hctsa['Value']), bins=geometric_sequence, edgecolor='white', alpha=0.8)
ax.set_xscale('log')
ax.set_xlabel('abs(Global average)')
ax.set_ylabel('Counts')
ax.axvline(threshold, color='red', linestyle='--')

fig.tight_layout()

In [ ]:
# number of zero operators
num_zero_ops = len(df_ops_zero)
print(f'Number of zero operators: {num_zero_ops}')

In [ ]:
# Write to Excel with formatting

# Create the save directory if it doesn't exist
os.makedirs(dir_supp, exist_ok=True)


with pd.ExcelWriter(dir_supp+'Supplement_Table_S1.xlsx', engine='xlsxwriter') as writer:
    df_ops_zero.to_excel(writer, index=False, sheet_name='Sheet1')

    workbook  = writer.book
    worksheet = writer.sheets['Sheet1']

    # Define formats
    format_grey = workbook.add_format({'bg_color': '#DDDDDD'})
    format_white = workbook.add_format({'bg_color': '#FFFFFF'})

    # Specify the header names in bold
    bold_format = workbook.add_format({'bold': True, 'font_color': '#000000', 'bg_color': '#DDDDDD'})
    header_names = ['Name', 'Global average', 'Feature class', 'Accuracy']
    # Write the header with bold format
    for col_num, header in enumerate(df_ops_zero.columns):
        worksheet.write(0, col_num, header, bold_format)


    # Apply alternating colors row by row
    for row in range(1, len(df_ops_zero)+1):  # Start at 1 to skip header
        row_format = format_grey if row % 2 == 0 else format_white
        worksheet.set_row(row, 20, row_format)
    worksheet.set_column('A:A', 40)  # Set width for the first column
    worksheet.set_column('B:B', 20)  # Set width for the second column
    worksheet.set_column('C:C', 20)  # Set width for the third column